<a href="https://colab.research.google.com/github/Umama123/Machine-Learning-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**ML Task Type**:  Supervised Machine Learning — Scoring & Learning-to-Rank (Regression).

**Framing:**
 Instead of predicting a binary outcome (Yes/No), we frame this as a continuous Refresh Opportunity Scoring problem (0 to 100). The model takes historical performance signals and generates a scalar priority score for each page, sorting them into a ranked review queue for the SEO team.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target / Proxy Definition:** Since future organic growth after a refresh cannot be directly measured in raw historical data, we create a composite Performance Decay Proxy Target.

**Proxy Metric:** A score combining the trailing drop in impressions (impressions_30d vs impressions_90d), rank slippage (position_avg movement), and total available search volume. Pages with high total potential demand but severe short-term drop receive the highest target priority.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**ML Offline Metrics:**

NDCG@K (Normalized Discounted Cumulative Gain at Top K): Evaluates whether the pages that need a refresh most urgently are actually ranked at the very top of our priority list.

**MAE / RMSE:** Measures error in target decay score prediction.

Business Success Metric: Traffic Recovery per Editor Hour — Measuring the percentage of lost organic impressions recovered after editors refresh pages recommended by the top of our queue vs. standard guessing.

In [3]:
import os

# Agar repo downloaded nahi hai to clone karega, varna folder mein chala jayega
if not os.path.exists('/content/Machine-Learning-Internship'):
    !git clone https://github.com/Umama123/Machine-Learning-Internship.git

# Main repository folder mein move karke git pull karein
%cd /content/Machine-Learning-Internship
!git pull

# Dobara notebooks wale folder mein jayein
%cd work/notebooks

Cloning into 'Machine-Learning-Internship'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 111 (delta 28), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 1.84 MiB | 8.47 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/Machine-Learning-Internship
Already up to date.
/content/Machine-Learning-Internship/work/notebooks


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** 1 Row = 1 Unique Content Page (content_id). Each row represents the aggregate metrics and trend status for an individual URL/content piece in the inventory.

**Target Column** (target_refresh_priority): A continuous priority score ranging from 0 to 100.

Pages with trend_direction == 'down' receive a scaled score based on their impression volume at risk (higher impressions lost = score closer to 100).

Stable or growing pages (trend_direction == 'up' or 'flat') receive a target score of 0, ensuring the model focuses exclusively on content refresh opportunities.

In [5]:
import pandas as pd
import numpy as np

# 1. Load starter dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Print all available columns to verify exact names
print("Available columns in dataset:")
print(df.columns.tolist())

# Dynamically find impression and position columns
imp_cols = [c for c in df.columns if 'impression' in c]
pos_cols = [c for c in df.columns if 'position' in c or 'pos' in c or 'rank' in c]

# Select available unit of analysis columns
selected_cols = [col for col in ['content_id', 'trend_direction'] + imp_cols + pos_cols if col in df.columns]
unit_of_analysis = df[selected_cols].copy()

# 2. Sketch Target Priority Score (0 to 100)
# Use primary impression column for weighting decay risk
primary_imp = imp_cols[0] if imp_cols else None

if primary_imp:
    max_imp = unit_of_analysis[primary_imp].max()
    unit_of_analysis['target_refresh_priority'] = np.where(
        unit_of_analysis['trend_direction'] == 'down',
        np.clip((unit_of_analysis[primary_imp] / max_imp) * 100, 15.0, 100.0),
        0.0
    )
else:
    unit_of_analysis['target_refresh_priority'] = np.where(
        unit_of_analysis['trend_direction'] == 'down', 50.0, 0.0
    )

# Round target score to 2 decimal places and sort by highest priority
unit_of_analysis['target_refresh_priority'] = unit_of_analysis['target_refresh_priority'].round(2)
unit_of_analysis = unit_of_analysis.sort_values(by='target_refresh_priority', ascending=False).reset_index(drop=True)

print("\n" + "="*50)
print(f"Unit of Analysis: 1 Row = 1 Unique Content Page (`content_id`)")
print(f"Total Pages Analyzed: {len(unit_of_analysis):,}")
print("="*50)
print("\n--- Preview of Unit of Analysis Dataframe ---")
unit_of_analysis.head(10)

Available columns in dataset:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Unit of Analysis: 1 Row = 1 Unique Content Page (`content_id`)
Total Pages Analyzed: 30,000

--- Preview of Unit of Analysis Dataframe ---


,content_id,trend_direction,impressions_90d,days_with_impressions,impressions_last_30d,impressions_prev_30d,impression_tier,avg_position,position_tier,target_refresh_priority
0,content_5fe46e04994d,down,517715,88,120791,218786,excellent,4.2,page_1,100.00
1,content_8c19996aa890,down,509252,88,89463,161284,excellent,2.5,top_3,98.37
2,content_4c36c775b818,down,463103,88,83723,125416,excellent,2.3,top_3,89.45
3,content_1a9e894be2e2,down,416180,88,107986,147889,excellent,4.0,page_1,80.39
4,content_2c2606c5d176,down,347399,88,104248,164079,excellent,4.2,page_1,67.10
5,content_cb112fce36be,down,309910,88,72468,124500,excellent,5.6,page_1,59.86
6,content_9532f197bbc8,down,309192,88,109317,174235,excellent,2.0,top_3,59.72
7,content_008fb02c46cb,down,236803,88,66042,83932,excellent,4.4,page_1,45.74
8,content_813e88069237,down,233561,88,62744,94762,excellent,26.2,page_3_5,45.11
9,content_ff94c9b6b411,down,228566,88,65011,96122,excellent,27.4,page_3_5,44.15


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Nuanced Multi-Variable Interaction:** Simple business rules (e.g., "Flag any page where traffic drops by 20%") are too rigid. They fail to account for the interplay between content age, intent type, keyword competition, and position movement. An ML model learns non-linear relationships across all these features simultaneously.

**Prioritization vs. Categorization:**  A fixed rule creates huge, unranked buckets of hundreds of "decayed" pages without telling editors which ones to fix first. ML provides a continuous priority score (0–100), allowing content teams to sort pages into an actionable review queue and focus editorial effort where search volume risk is highest.

**Handling Noise & Seasonality:** Machine learning models adapt to broad baseline shifts across 30,000+ pages without triggering false alarms for minor, temporary fluctuations that rigid threshold rules often misclassify.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.